# TASK 2: Exploratory Data Analysis (EDA)

In [1]:
#Goal: Explore structure, detect patterns & anomalies, test hypotheses.
#Tools: `pandas`, `numpy`, `scipy.stats`

In [3]:
# Import required libraries
# requests  : used to send HTTP requests to the website
# BeautifulSoup : used to parse and extract HTML content
# pandas    : used for data storage and manipulation
# numpy     : used for generating synthetic (random) data
# time      : used to add delay between requests (ethical scraping)

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time

# Dictionary to convert textual star ratings into numeric values
# Example: "Three" → 3
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# HTTP headers with a User-Agent to identify the request
# This helps avoid blocking and follows ethical scraping practices
HEADERS = {
    "User-Agent": "Mozilla/5.0 (educational scraping project)"
}


# Function to scrape book data from books.toscrape.com
def scrape_books(pages=5):
    """
    Scrapes book information from multiple pages of books.toscrape.com.

    Parameters:
    pages (int): Number of pages to scrape

    Returns:
    DataFrame: Pandas DataFrame containing scraped book data
    """

    books_data = []  # List to store scraped book information

    # Loop through each page
    for page_num in range(1, pages + 1):

        # Construct page URL dynamically
        url = f"http://books.toscrape.com/catalogue/page-{page_num}.html"

        # Send GET request to the website
        response = requests.get(url, headers=HEADERS, timeout=10)

        # Raise an error if the request fails (e.g., 404 or 500)
        response.raise_for_status()

        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, "html.parser")

        # Select all book containers on the page
        for book in soup.select("article.product_pod"):

            # Extract book title
            title = book.h3.a["title"]

            # Extract and clean book price (remove currency symbol)
            price_text = book.select_one(".price_color").text.strip()
            price = float(''.join(c for c in price_text if c.isdigit() or c == '.'))

            # Extract star rating and convert it to numeric form
            rating_text = book.p["class"][1]
            rating = RATING_MAP.get(rating_text, 0)

            # Extract availability status
            availability = book.select_one(".availability").text.strip()

            # Store extracted data in dictionary format
            books_data.append({
                "Title": title,
                "Price_GBP": price,
                "Rating": rating,
                "Availability": availability,
                "Page": page_num
            })

        # Print progress message for each page
        print(f"Page {page_num} scraped successfully")

        # Delay between requests to avoid overloading the server
        time.sleep(0.5)

    # Convert the collected data into a Pandas DataFrame
    return pd.DataFrame(books_data)


# Function to generate a synthetic dataset if live scraping fails
def generate_synthetic_books(n=100):
    """
    Generates a synthetic book dataset for offline use or testing.

    Parameters:
    n (int): Number of synthetic records to generate

    Returns:
    DataFrame: Pandas DataFrame with generated book data
    """

    # Set random seed for reproducibility
    np.random.seed(42)

    # Sample genres and authors
    genres = [
        "Fiction", "Mystery", "Sci-Fi", "Romance", "History",
        "Biography", "Self-Help", "Thriller", "Fantasy", "Horror"
    ]

    authors = [
        "Alice Hartman", "Bob Chen", "Clara Voss", "David Kim",
        "Eva Martinez", "Frank O'Neill", "Grace Lee", "Henry Park"
    ]

    # Randomly generate data for each column
    genre_col = np.random.choice(genres, n)
    titles = [f"The {g} Chronicles Vol.{i+1}" for i, g in enumerate(genre_col)]
    ratings = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.10, 0.20, 0.35, 0.30])
    prices = np.round(np.random.uniform(5, 55, n), 2)
    availability = np.random.choice(["In stock", "Out of stock"], size=n, p=[0.85, 0.15])

    # Create and return the DataFrame
    return pd.DataFrame({
        "Title": titles,
        "Price_GBP": prices,
        "Rating": ratings,
        "Availability": availability,
        "Genre": genre_col,
        "Author": np.random.choice(authors, n),
        "Page": np.random.randint(1, 6, n)
    })


# Main execution block
print("Attempting live web scrape from books.toscrape.com...")

try:
    # Try to scrape live data
    df_books = scrape_books(pages=5)
    print(f"\nLive scraping SUCCESS - {len(df_books)} books collected")

except Exception as e:
    # If scraping fails, generate synthetic data instead
    print(f"Live scraping unavailable ({type(e).__name__}). Using synthetic dataset...")
    df_books = generate_synthetic_books(n=100)
    print(f"Synthetic dataset generated - {len(df_books)} books")

# Save the final dataset to a CSV file
df_books.to_csv("scraped_books.csv", index=False)

# Display dataset information
print(f"\nDataset saved: scraped_books.csv | Shape: {df_books.shape}")

# Display first 8 records for verification
df_books.head(8)

Attempting live web scrape from books.toscrape.com...
Page 1 scraped successfully
Page 2 scraped successfully
Page 3 scraped successfully
Page 4 scraped successfully
Page 5 scraped successfully

Live scraping SUCCESS - 100 books collected

Dataset saved: scraped_books.csv | Shape: (100, 5)


,Title,Price_GBP,Rating,Availability,Page
0,A Light in the Attic,51.77,3,In stock,1
1,Tipping the Velvet,53.74,1,In stock,1
2,Soumission,50.10,1,In stock,1
3,Sharp Objects,47.82,4,In stock,1
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,1
5,The Requiem Red,22.65,1,In stock,1
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,1
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,1


# TASK 2: Exploratory Data Analysis (EDA) starts from here....

In [9]:
# Import required libraries
# pandas : used for data loading and analysis
# numpy  : used for numerical operations (optional here but commonly used in EDA)
# scipy.stats : used for statistical analysis (can be extended later)

import pandas as pd
import numpy as np
from scipy import stats


# Load the dataset from the CSV file into a Pandas DataFrame
df = pd.read_csv("scraped_books.csv")


# Print a separator line for better readability
print("=" * 60)

# Print the title of the Exploratory Data Analysis (EDA) report
print("        EXPLORATORY DATA ANALYSIS REPORT")

# Print another separator line
print("=" * 60)


# Display the shape of the dataset (number of rows and columns)
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")


# Display data types of each column
# Helps identify numeric, categorical, and text data
print("\nData Types:")
print(df.dtypes)


# Check for missing (null) values in each column
print("\nMissing Values:")
missing = df.isnull().sum()

# Print only columns that have missing values
if missing.any():
    print(missing[missing > 0])
else:
    print("  None found")


# Check and print the number of duplicate rows in the dataset
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

        EXPLORATORY DATA ANALYSIS REPORT

Shape: 100 rows x 5 columns

Data Types:
Title            object
Price_GBP       float64
Rating            int64
Availability     object
Page              int64
dtype: object

Missing Values:
  None found

Duplicate Rows: 0


In [10]:
# Print a heading for the descriptive statistics section
print("Descriptive Statistics:")

# Generate descriptive statistics for all numerical columns
# This includes count, mean, standard deviation, min, max,
# and quartile values (25%, 50%, 75%)
# round(2) is used to limit the output to 2 decimal places for readability
df.describe().round(2)

Descriptive Statistics:


,Price_GBP,Rating,Page
count,100.00,100.00,100.00
mean,34.56,2.93,3.00
std,14.64,1.42,1.42
min,10.16,1.00,1.00
25%,19.90,2.00,2.00
50%,34.78,3.00,3.00
75%,47.97,4.00,4.00
max,58.11,5.00,5.00


In [11]:
# -------------------------------
# Outlier Detection using IQR Method
# -------------------------------

# Calculate the first quartile (25th percentile) of book prices
Q1 = df['Price_GBP'].quantile(0.25)

# Calculate the third quartile (75th percentile) of book prices
Q3 = df['Price_GBP'].quantile(0.75)

# Compute the Interquartile Range (IQR)
# IQR = Q3 - Q1
IQR = Q3 - Q1

# Define the lower bound for detecting outliers
# Any value below this is considered an outlier
lower_bound = Q1 - 1.5 * IQR

# Define the upper bound for detecting outliers
# Any value above this is considered an outlier
upper_bound = Q3 + 1.5 * IQR

# Filter rows where the price lies outside the normal range
outliers = df[
    (df['Price_GBP'] < lower_bound) |
    (df['Price_GBP'] > upper_bound)
]


# Print the heading for outlier detection results
print("Outlier Detection - Price (IQR Method):")

# Print quartile and IQR values for clarity
print(f"  Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}")

# Print the normal price range calculated using IQR
print(f"  Normal range: {lower_bound:.2f} to {upper_bound:.2f}")

# Print the number of outliers detected
print(f"  Outliers found: {len(outliers)}")

# If outliers exist, display their titles and prices
if not outliers.empty:
    print(outliers[['Title', 'Price_GBP']].to_string(index=False))

Outlier Detection - Price (IQR Method):
  Q1=19.90  Q3=47.97  IQR=28.07
  Normal range: -22.21 to 90.07
  Outliers found: 0


In [12]:
# -----------------------------------------
# Hypothesis Testing: Price vs Book Ratings
# -----------------------------------------

# Separate book prices into two groups based on ratings
# High-rated books: Rating 4 or 5
high = df[df['Rating'] >= 4]['Price_GBP']

# Low-rated books: Rating 1 or 2
low = df[df['Rating'] <= 2]['Price_GBP']


# Perform an Independent Two-Sample T-Test
# This test checks whether the mean prices of two independent groups
# (high-rated and low-rated books) are significantly different
t_stat, p_value = stats.ttest_ind(high, low)


# Print heading for hypothesis testing results
print("Hypothesis Test - Independent T-Test:")

# State the null hypothesis (H0)
print("  H0: High-rated and low-rated books have the same avg price")

# State the alternative hypothesis (H1)
print("  H1: There is a significant price difference")

# Print average price and sample size for high-rated books
print(f"  High-rated avg: GBP {high.mean():.2f}  (n={len(high)})")

# Print average price and sample size for low-rated books
print(f"  Low-rated avg : GBP {low.mean():.2f}  (n={len(low)})")

# Print the calculated t-statistic and p-value
print(f"  T-stat: {t_stat:.4f}  |  P-value: {p_value:.4f}")


# Interpret the result based on significance level (alpha = 0.05)
# If p-value < 0.05 → Reject the null hypothesis
result = (
    "Reject H0 - significant difference"
    if p_value < 0.05
    else "Fail to reject H0 - no significant difference"
)

# Print the final conclusion of the hypothesis test
print(f"  Conclusion: {result}")

# -----------------------------------------
# Correlation Analysis: Price vs Rating
# -----------------------------------------
# Calculate Pearson correlation coefficient
# This measures the strength and direction of linear relationship
# between book price and rating
corr, pval = stats.pearsonr(df['Rating'], df['Price_GBP'])

# Print correlation results
print(f"\nPearson Correlation (Price vs Rating): r={corr:.4f}  p={pval:.4f}")

Hypothesis Test - Independent T-Test:
  H0: High-rated and low-rated books have the same avg price
  H1: There is a significant price difference
  High-rated avg: GBP 31.94  (n=37)
  Low-rated avg : GBP 35.70  (n=41)
  T-stat: -1.1573  |  P-value: 0.2508
  Conclusion: Fail to reject H0 - no significant difference

Pearson Correlation (Price vs Rating): r=-0.1217  p=0.2276


In [13]:
# -----------------------------------------
# Key Insights Derived from Exploratory Data Analysis
# -----------------------------------------

# Print a heading for the key insights section
print("Key Insights from EDA:")

# Identify and print the most expensive book
# idxmax() gives the index of the highest price
# Title is truncated to 55 characters for neat display
print(
    f"  Most expensive: "
    f"'{df.loc[df['Price_GBP'].idxmax(), 'Title'][:55]}' "
    f"@ {df['Price_GBP'].max():.2f}"
)

# Identify and print the cheapest book
# idxmin() gives the index of the lowest price
print(
    f"  Cheapest      : "
    f"'{df.loc[df['Price_GBP'].idxmin(), 'Title'][:55]}' "
    f"@ {df['Price_GBP'].min():.2f}"
)

# Calculate and print the number and percentage of 5-star rated books
print(
    f"  5-star books  : "
    f"{(df['Rating'] == 5).sum()} "
    f"({(df['Rating'] == 5).mean() * 100:.1f}%)"
)

# Count and print how many books are currently in stock
print(
    f"  In-stock books: "
    f"{(df['Availability'] == 'In stock').sum()}"
)

# Calculate and print skewness of book prices
# Skewness indicates whether the price distribution is left or right skewed
print(
    f"  Price skewness: "
    f"{df['Price_GBP'].skew():.3f}"
)

# Check if Genre column exists (important when using scraped vs synthetic data)
if 'Genre' in df.columns:

    # Find the most frequent genre
    top_genre = df['Genre'].value_counts().idxmax()

    # Print the most common genre and its count
    print(
        f"  Top genre     : "
        f"{top_genre} "
        f"({df['Genre'].value_counts().max()} books)"
    )

Key Insights from EDA:
  Most expensive: 'The Death of Humanity: and the Case for Life' @ 58.11
  Cheapest      : 'Patience' @ 10.16
  5-star books  : 19 (19.0%)
  In-stock books: 100
  Price skewness: 0.043
